# Apply wavelength calibration to processed mosaic
Checks each raster individually to check if orbital variation is influencing it

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
# %matplotlib notebook

import pathlib as pl
from iris_mosaics import MosaicConfig, plan_rasters, read_pointing
import numpy as np
from skimage.filters import threshold_otsu
import pickle
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.pyplot as plt
from matplotlib import colors
# params = {"ytick.color" : "k",
#           "xtick.color" : "k",
#           "axes.labelcolor" : "k",
#           "axes.edgecolor" : "k"}
# plt.rcParams.update(params)
import astropy.units as u

from iris_mosaics import read_sg_image, build_mosaic_single_wavelength
import iris_mosaics as iris_fdm
from astropy.modeling import models, fitting
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support
quantity_support()

#### Background-subtracted data file path

In [ ]:
# Path to selected mosaic
cfg = MosaicConfig.load('20240811')   # <-- the only per-mosaic edit needed
path = cfg.data_root
# JPW's + MSU background-subtracted
path_level_15 = path / 'level_15'
files_15 = list(path_level_15.glob('*.fits'))

# # JPW's new background subtraction method only
# path_level_15_JPW = path / 'JP_new_bg_subtracted' / 'level_15_new_iris_bgsub'
# files_15_JPW = list(path_level_15_JPW.glob('*.fits'))

In [ ]:
%%time
# Load images
sg_wcs = []
sg_img = []

for i, file in enumerate(files_15):

    w, hdu, _ = read_sg_image(file,'fuv2')
    img = hdu[0].data

    sg_wcs.append(w)
    sg_img.append(img)

sg_img_set = np.array(sg_img)

In [ ]:
sg_img_set[sg_img_set == np.inf] = np.nan

In [ ]:
sg_img_set_mean = np.nanmean(sg_img_set, axis=0)

Use the code below to apply nanmean when memory runs out (helpful for those mosaics with twice the pixels...)

In [ ]:
# Kept running out of memory trying to generate the nanmean image... this does it in "chunks"
shape = sg_img_set.shape[1:]                    # (548, 1036)
acc = np.zeros(shape, dtype=np.float64)
cnt = np.zeros(shape, dtype=np.int64)

chunk = 500                                  # frames per chunk; tune to taste
for i0 in range(0, sg_img_set.shape[0], chunk):
    block = sg_img_set[i0:i0 + chunk]
    good = ~np.isnan(block)
    acc += np.where(good, block, 0).sum(axis=0, dtype=np.float64)
    cnt += good.sum(axis=0)

sg_img_set_mean = np.divide(acc, cnt, out=np.full(shape, np.nan), where=cnt > 0)

In [ ]:
plt.figure(figsize=(15,5))
plt.imshow(sg_img_set_mean,
           vmax=np.nanpercentile(sg_img_set_mean,99.9),
           vmin=np.nanpercentile(sg_img_set_mean,0.1),
           origin='lower',)
plt.colorbar(pad=0.01).set_label('DN')
plt.xticks([])
plt.yticks([])

# Save mean image as png
# plt.savefig(path / 'sg_img_15_mean.png',dpi=300, transparent=True, bbox_inches='tight')

# Save mean image as pickle file
# with open(path / 'sg_img_15_mean.pickle', 'wb') as fh:
#     pickle.dump(sg_img_set_mean, fh)

In [ ]:
wcs_15, hdu_15, _ = read_sg_image(files_15[0],'fuv2')
img_15 = hdu_15[0].data

sg_wavelength_full = np.squeeze(wcs_15.array_index_to_world(*np.indices((1, 1, img_15.shape[1])))[0].to(u.Angstrom))

In [ ]:
spectrum_mean = np.nanmean(sg_img_set_mean, axis=0)

In [ ]:
# For regular deep mosaics...
# sl_1394 = slice(None, None), slice(None, None), slice(735,810)
# sl_1403 = slice(None, None), slice(None, None), slice(870,1023)
#
# sl_1394_wl = slice(735,810)
# sl_1403_wl = slice(870,1023)

# For the Aug 2024 mosaic that is twice as wide in the x direction...
x1 = 2 * 735
x2 = 2 * 810
x3 = 2 * 870
x4 = 2 * 1023
sl_1394 = slice(None, None), slice(None, None), slice(x1,x2)
sl_1403 = slice(None, None), slice(None, None), slice(x3,x4)

sl_1394_wl = slice(x1,x2)
sl_1403_wl = slice(x3,x4)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(sg_wavelength_full[sl_1394_wl], spectrum_mean[sl_1394_wl])
plt.plot(sg_wavelength_full[sl_1403_wl], spectrum_mean[sl_1403_wl], color='tab:blue')
plt.plot(sg_wavelength_full[730:], spectrum_mean[730:], color='grey', ls='--', lw=1, alpha=0.5)
plt.axhline(0, color='red', lw=1, ls=':')

In [ ]:
sg_img_set_1394 = sg_img_set[sl_1394]
sg_img_set_1403 = sg_img_set[sl_1403]

In [ ]:
sg_wavelength_1394 = sg_wavelength_full[sl_1394_wl]
sg_wavelength_1403 = sg_wavelength_full[sl_1403_wl]

In [ ]:
del sg_img_set

### Reshape into sets of rasters so we can check each individually

Same raster layout logic as `apply_rolling_trimmed_mean` — it lives in
`iris_mosaics.rasters` so both notebooks share one implementation.


In [ ]:
%%time
solar_x, solar_y, t_obs = read_pointing(files_15)


In [ ]:
layout = plan_rasters(
    solar_x,
    num_img_per_raster=cfg.num_img_per_raster,
    step_arcsec=cfg.raster_step.value,
)
num_img_per_raster = layout.num_img_per_raster

print(f'{layout.num_images_original} images -> {layout.num_rasters} rasters of {num_img_per_raster}')
print(f'inserted {layout.inserted.sum()} NaN frames ({layout.missing_image_index.size} in-raster gaps, {layout.raster_missing_end_index.size} short rasters)')


In [ ]:
sg_img_set_1394_rasters = layout.to_rasters(layout.pad(sg_img_set_1394))
sg_img_set_1403_rasters = layout.to_rasters(layout.pad(sg_img_set_1403))

print(sg_img_set_1394_rasters.shape)   # (raster, image, y, x)


### Check each raster individually

In [ ]:
sg_img_raster_mean_1394 = np.nanmean(sg_img_set_1394_rasters, axis=1)
sg_img_raster_mean_1403 = np.nanmean(sg_img_set_1403_rasters, axis=1)

In [ ]:
sg_spectrum_raster_mean_1394 = np.nanmean(sg_img_raster_mean_1394, axis=1)
sg_spectrum_raster_mean_1403 = np.nanmean(sg_img_raster_mean_1403, axis=1)

In [ ]:
plt.figure(figsize=(9,7))
for spectrum in sg_spectrum_raster_mean_1394:
    plt.plot(sg_wavelength_1394, spectrum)
# Neutral lines (plus Fe II)
line_list_1394 = [1392.149, 1392.588, 1392.817] * u.AA
label_list_1394 = ['Fe II', 'S I', 'Fe II']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)
plt.ylim((-0.2,1.7))
plt.xlim((1391.8*u.AA, 1393.15*u.AA))

In [ ]:
plt.figure(figsize=(9,7))
for spectrum in sg_spectrum_raster_mean_1403:
    plt.plot(sg_wavelength_1403, spectrum)
# Neutral lines
line_list_1394 = [1401.5136] * u.AA
label_list_1394 = ['S I']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)
plt.ylim((-0.25,6))
plt.xlim((1400.75*u.AA, 1401.75*u.AA))

For 2024-08-11

In [ ]:
from astropy.modeling import models, fitting

# Vacuum wavelengths
fe_ii_wl_1 = 1392.149
s_i_wl = 1392.588
fe_ii_wl_2 = 1392.817

obvious_offset = 0.02

model_init_nl = (models.Gaussian1D(amplitude=0.2, mean=fe_ii_wl_1 + obvious_offset, stddev=0.04, name="FeII_1") +
                models.Gaussian1D(amplitude=0.1, mean=s_i_wl + obvious_offset, stddev=0.04, name="SI") +
                models.Gaussian1D(amplitude=0.5, mean=fe_ii_wl_2 + obvious_offset, stddev=0.04, name="FeII_2"))

model_init_nl.mean_0.bounds = (fe_ii_wl_1 + obvious_offset - 0.02, fe_ii_wl_1 + obvious_offset + 0.02)
model_init_nl.mean_1.bounds = (s_i_wl + obvious_offset - 0.02, s_i_wl + obvious_offset + 0.02)
model_init_nl.mean_2.bounds = (fe_ii_wl_2 + obvious_offset - 0.02, fe_ii_wl_2 + obvious_offset + 0.02)

model_init_nl.stddev_0.bounds = (0.02, 0.06)
model_init_nl.stddev_1.bounds = (0.02, 0.06)
model_init_nl.stddev_2.bounds = (0.02, 0.06)

# Create a mask for finite values
mask = np.isfinite(sg_wavelength_full) & np.isfinite(spectrum_mean)

# Apply the mask to data
clean_wavelength = sg_wavelength_full[mask]
clean_flux = spectrum_mean[mask]

# Fit the data
fitter = fitting.LevMarLSQFitter()

best_fit_nl = fitter(model_init_nl, clean_wavelength, clean_flux)

print('true - best fit mean: ', fe_ii_wl_1 - best_fit_nl.mean_0.value, s_i_wl - best_fit_nl.mean_1.value, fe_ii_wl_2 - best_fit_nl.mean_2.value)
print('mean: ', best_fit_nl.mean_0.value, best_fit_nl.mean_1.value, best_fit_nl.mean_2.value)
print('std dev: ', best_fit_nl.stddev_0.value, best_fit_nl.stddev_1.value, best_fit_nl.stddev_2.value)
print('amplitude: ', best_fit_nl.amplitude_0.value, best_fit_nl.amplitude_1.value, best_fit_nl.amplitude_2.value)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(sg_wavelength_full, spectrum_mean)
plt.plot(sg_wavelength_full, best_fit_nl(sg_wavelength_full), ls='dashed')
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
line_list_1394 = [1392.149, 1392.588, 1392.817] * u.AA
label_list_1394 = ['Fe II', 'S I', 'Fe II']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)

# Fit wavelengths
plt.axvline(x=best_fit_nl.mean_0.value * u.AA, color='tab:orange', linewidth=1, linestyle='dotted', label='best fit')
plt.axvline(x=best_fit_nl.mean_1.value * u.AA, color='tab:orange', linewidth=1, linestyle='dotted')
plt.axvline(x=best_fit_nl.mean_2.value * u.AA, color='tab:orange', linewidth=1, linestyle='dotted')
plt.legend()

plt.ylabel('mean full disk spectrum [DN]')
plt.ylim((-0.2,1.2))
plt.xlim((1392*u.AA, 1393*u.AA))

print('Fe II 1392.1 true - best fit = ', line_list_1394[0] - best_fit_nl.mean_0.value * u.AA)
print('S I 1392.6 true - best fit = ', line_list_1394[1] - best_fit_nl.mean_1.value * u.AA)
print('Fe II 1392.8 true - best fit = ', line_list_1394[2] - best_fit_nl.mean_2.value * u.AA)
print('mean = ', ((line_list_1394[0] - best_fit_nl.mean_0.value * u.AA) +
      (line_list_1394[1] - best_fit_nl.mean_1.value * u.AA) +
      (line_list_1394[2] - best_fit_nl.mean_2.value * u.AA)) / 3)

For 2014-03-24

In [ ]:
# from astropy.modeling import models, fitting
#
# # Vacuum wavelengths
# fe_ii_wl_1 = 1392.149
# s_i_wl = 1392.588
# fe_ii_wl_2 = 1392.817
#
# obvious_offset = -0.011
#
# model_init_nl = (models.Gaussian1D(amplitude=0.5, mean=fe_ii_wl_1 + obvious_offset, stddev=0.04, name="FeII_1") +
#                 models.Gaussian1D(amplitude=0.3, mean=s_i_wl + obvious_offset, stddev=0.04, name="SI") +
#                 models.Gaussian1D(amplitude=1.1, mean=fe_ii_wl_2 + obvious_offset, stddev=0.04, name="FeII_2"))
#
# model_init_nl.mean_0.bounds = (fe_ii_wl_1 + obvious_offset - 0.02, fe_ii_wl_1 + obvious_offset + 0.02)
# model_init_nl.mean_1.bounds = (s_i_wl + obvious_offset - 0.02, s_i_wl + obvious_offset + 0.02)
# model_init_nl.mean_2.bounds = (fe_ii_wl_2 + obvious_offset - 0.02, fe_ii_wl_2 + obvious_offset + 0.02)
#
# model_init_nl.stddev_0.bounds = (0.01, 0.05)
# model_init_nl.stddev_1.bounds = (0.01, 0.05)
# model_init_nl.stddev_2.bounds = (0.01, 0.05)
#
# # Create a mask for finite values
# mask = np.isfinite(sg_wavelength_full) & np.isfinite(spectrum_mean)
#
# # Apply the mask to data
# clean_wavelength = sg_wavelength_full[mask]
# clean_flux = spectrum_mean[mask]
#
# # Fit the data
# fitter = fitting.LevMarLSQFitter()
#
# best_fit_nl = fitter(model_init_nl, clean_wavelength, clean_flux)
#
# print('true - best fit mean: ', fe_ii_wl_1 - best_fit_nl.mean_0.value, s_i_wl - best_fit_nl.mean_1.value, fe_ii_wl_2 - best_fit_nl.mean_2.value)
# print('mean: ', best_fit_nl.mean_0.value, best_fit_nl.mean_1.value, best_fit_nl.mean_2.value)
# print('std dev: ', best_fit_nl.stddev_0.value, best_fit_nl.stddev_1.value, best_fit_nl.stddev_2.value)
# print('amplitude: ', best_fit_nl.amplitude_0.value, best_fit_nl.amplitude_1.value, best_fit_nl.amplitude_2.value)

For 2019-09-12

In [ ]:
# from astropy.modeling import models, fitting
#
# # Vacuum wavelengths
# fe_ii_wl_1 = 1392.149
# s_i_wl = 1392.588
# fe_ii_wl_2 = 1392.817
#
# model_init_nl = (models.Gaussian1D(amplitude=0.2, mean=fe_ii_wl_1 - 0.02, stddev=0.02, name="FeII_1") +
#                 models.Gaussian1D(amplitude=0.2, mean=s_i_wl - 0.02, stddev=0.03, name="SI") +
#                 models.Gaussian1D(amplitude=0.6, mean=fe_ii_wl_2 - 0.02, stddev=0.04, name="FeII_2"))
#
# model_init_nl.mean_0.bounds = (fe_ii_wl_1 - 0.02 + 0.014, fe_ii_wl_1 - 0.02 + 0.1)
# model_init_nl.mean_1.bounds = (s_i_wl - 0.02 - 0.001, s_i_wl - 0.02 + 0.009)
# model_init_nl.mean_2.bounds = (fe_ii_wl_2 - 0.02 - 0.00, fe_ii_wl_2 - 0.02 + 0.0087)
#
# model_init_nl.stddev_0.bounds = (0.01, 0.05)
# model_init_nl.stddev_1.bounds = (0.01, 0.05)
# model_init_nl.stddev_2.bounds = (0.01, 0.05)
#
# # Create a mask for finite values
# mask = np.isfinite(sg_wavelength_full) & np.isfinite(spectrum_mean)
#
# # Apply the mask to data
# clean_wavelength = sg_wavelength_full[mask]
# clean_flux = spectrum_mean[mask]
#
# # Fit the data
# fitter = fitting.LevMarLSQFitter()
#
# best_fit_nl = fitter(model_init_nl, clean_wavelength, clean_flux)
#
# print('true - best fit mean: ', fe_ii_wl_1 - best_fit_nl.mean_0.value, s_i_wl - best_fit_nl.mean_1.value, fe_ii_wl_2 - best_fit_nl.mean_2.value)
# print('mean: ', best_fit_nl.mean_0.value, best_fit_nl.mean_1.value, best_fit_nl.mean_2.value)
# print('std dev: ', best_fit_nl.stddev_0.value, best_fit_nl.stddev_1.value, best_fit_nl.stddev_2.value)
# print('amplitude: ', best_fit_nl.amplitude_0.value, best_fit_nl.amplitude_1.value, best_fit_nl.amplitude_2.value)

In [ ]:
sg_wavelength_neutral_line_aligned = sg_wavelength_full + (-0.018*u.AA)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(sg_wavelength_neutral_line_aligned, spectrum_mean)
plt.plot(sg_wavelength_neutral_line_aligned, best_fit_nl(sg_wavelength_full), ls='dashed')
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
line_list_1394 = [1392.149, 1392.588, 1392.817] * u.AA
label_list_1394 = ['Fe II', 'S I', 'Fe II']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)

plt.ylabel('mean full disk spectrum [DN]')
plt.ylim((-0.2,1.2))
plt.xlim((1392*u.AA, 1393*u.AA))

Check if the shift seems to fit well with the 1403 neutral line
First, original spectral wavelength...

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(sg_wavelength_full, spectrum_mean)
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
line_list_1394 = [1401.5136] * u.AA
label_list_1394 = ['S I']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)

plt.ylabel('mean full disk spectrum [DN]')
plt.ylim((-0.2,1))
plt.xlim((1401*u.AA, 1402*u.AA))
plt.title('Before $\lambda$ shift');

Then with the shifted spectrum...

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(sg_wavelength_neutral_line_aligned, spectrum_mean)
plt.axhline(y=0, color='r', linewidth=1, linestyle='dotted')
line_list_1394 = [1401.5136] * u.AA
label_list_1394 = ['S I']
for line, label in zip(line_list_1394, label_list_1394):
    plt.axvline(x=line, color='gray', linewidth=1, ls='dotted')
    plt.text(x=line, y=0.8, s=label, rotation='vertical', ha='right', size=9)

plt.ylabel('mean full disk spectrum [DN]')
plt.ylim((-0.2,1))
plt.xlim((1401*u.AA, 1402*u.AA))
plt.title('After $\lambda$ shift');

#### Shifts to apply to wavelength array:
- 2019-09-12: 0.011 Å
- 2014-03-24: 0.015 Å
- 2024-08-11: -0.018 Å